### Silver Discharge

In [1]:
file_path ='abfss://d5d4a0bc-04c6-4ae3-ae88-3b02cfbf37a8@onelake.dfs.fabric.microsoft.com/e9762736-9ad0-42b8-a949-2997f0bb5338/Tables/Bronze/bronzetable'

StatementMeta(, 9be6b2ff-46d7-47be-8911-3668e333f598, 3, Finished, Available, Finished, False)

In [3]:
df = spark.read.format('delta').load(file_path)
display(df)

StatementMeta(, 9be6b2ff-46d7-47be-8911-3668e333f598, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2db1942a-b932-4c1f-91cc-f31c806d3595)

In [4]:
df.columns

StatementMeta(, 9be6b2ff-46d7-47be-8911-3668e333f598, 6, Finished, Available, Finished, False)

['Health_Service_Area',
 'Hospital_County',
 'Operating_Certificate_Number',
 'Facility_Id',
 'Facility_Name',
 'Age_Group',
 'Zip_Code_3_digits',
 'Gender',
 'Race',
 'Ethnicity',
 'Length_of_Stay',
 'Type_of_Admission',
 'Patient_Disposition',
 'Discharge_Year',
 'CCS_Diagnosis_Code',
 'CCS_Diagnosis_Description',
 'CCS_Procedure_Code',
 'CCS_Procedure_Description',
 'APR_DRG_Code',
 'APR_DRG_Description',
 'APR_MDC_Code',
 'APR_MDC_Description',
 'APR_Severity_of_Illness_Code',
 'APR_Severity_of_Illness_Description',
 'APR_Risk_of_Mortality',
 'APR_Medical_Surgical_Description',
 'Payment_Typology_1',
 'Payment_Typology_2',
 'Payment_Typology_3',
 'Birth_Weight',
 'Abortion_Edit_Indicator',
 'Emergency_Department_Indicator',
 'Total_Charges',
 'Total_Costs']

In [6]:
#check null or nan columns
from pyspark.sql.functions import when, col, isnan, count
df.select([
    count(when(col(c).isNull() | isnan(c),c)).alias(c)
    for c in ['Age_Group','Gender','Length_of_Stay', 'Total_Charges']


]).show()

StatementMeta(, 9be6b2ff-46d7-47be-8911-3668e333f598, 8, Finished, Available, Finished, False)

+---------+------+--------------+-------------+
|Age_Group|Gender|Length_of_Stay|Total_Charges|
+---------+------+--------------+-------------+
|        0|     0|             0|            0|
+---------+------+--------------+-------------+



In [12]:
#change data types and trim
from pyspark.sql.functions  import trim, regexp_replace, upper,cast

df_cleaned = df.\
                withColumn('Age_Group', trim(col('Age_Group'))).\
                withColumn('Gender',upper(col('Gender'))).\
                withColumn('Total_Charges', regexp_replace('Total_Charges','[$,""]',"").cast('double')).\
                withColumn('Total_Costs', regexp_replace('Total_Costs','[$,""]',"").cast('double')).\
                withColumn('Length_of_Stay', col('Length_of_Stay').cast('int')).\
                withColumn('Birth_Weight', col('Birth_Weight').cast('double'))
display(df_cleaned)


StatementMeta(, 9be6b2ff-46d7-47be-8911-3668e333f598, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 813fef95-b67b-4af9-a33d-f7d6271888f0)

In [15]:
# Add derived columns
from pyspark.sql.functions import round
tran_df = df_cleaned.withColumn('Cost_per_Day',round(col("Total_Charges")/col("Length_of_Stay"),1))
display(tran_df)

StatementMeta(, 9be6b2ff-46d7-47be-8911-3668e333f598, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ad965e88-f1de-4827-a446-06e3589efffc)

In [16]:
tran_df.write.format('delta').saveAsTable('LH_Discharge.Silver.silvertable')

StatementMeta(, 9be6b2ff-46d7-47be-8911-3668e333f598, 18, Finished, Available, Finished, False)